# Parent-Child (Parent Document) Retrieval

**The tension in chunking:**
- *Small* chunks embed cleanly and give **precise** similarity matches - but they may be too short
  for the LLM to actually answer from.
- *Large* chunks carry enough context to answer - but their embedding is a blurry average of many
  topics, so retrieval is **less precise**.

**Parent-Child retrieval gets both:** split each page into large **parent** chunks, split every
parent into small **child** chunks. Embed and search the **children**; but when a child matches,
hand the LLM the **whole parent** it came from.

```
search on:   child chunks  (small, precise embeddings)
give to LLM:  the parent chunk each matching child belongs to  (full context)
```

LangChain's `ParentDocumentRetriever` manages the child index (a vector store) and the parent
store (a key-value docstore) together.

**Document:** `small.pdf` - VIT University's FFCS (Fully Flexible Credit System) Academic
Regulations 4.0: abbreviations, FFCS features, version history, admissions (VITEEE, VITMEE,
V-SIGN / B.Des). Niche, institution-specific content the base LLM does not know.

**Runs on Google Colab** (or Kaggle) - CPU is fine. On Kaggle, `Settings -> Internet -> On`.

## 1. Install

`ParentDocumentRetriever` lives in the **`langchain`** package. `pip install -U langchain`
now pulls **LangChain 1.0**, which removed `langchain.retrievers` (and `langchain.chains`,
`langchain.load`). This notebook needs the **0.3** line, so we pin the whole family.

In [ ]:
!pip install -q "langchain>=0.3,<1.0" "langchain-core>=0.3,<1.0" "langchain-community>=0.3,<1.0" "langchain-text-splitters>=0.3,<1.0" "langchain-huggingface>=0.1,<1.0" "langchain-chroma>=0.1,<1.0" "langchain-google-genai>=2.0,<3.0"
!pip install -q sentence-transformers chromadb pypdf

> ### Restart the runtime now
> **Runtime -> Restart session** (Colab) / **Run -> Restart session** (Kaggle), then run
> every cell from the top. Without a restart the freshly pinned `langchain` may not be
> picked up and the import below can still fail.

Sanity check after the restart: importing `ParentDocumentRetriever` proves the 0.3 pin took effect.

In [ ]:
# verify the install (run AFTER restarting)
import langchain
from langchain.retrievers import ParentDocumentRetriever
print("langchain", langchain.__version__, "- ParentDocumentRetriever import OK")

## 2. Imports

In [ ]:
from langchain.retrievers import ParentDocumentRetriever              # in the `langchain` package
from langchain_core.stores import InMemoryStore                       # parent-chunk store
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

## 3. The LLM (Google Gemini)

In [ ]:
import os
import warnings

# chromadb telemetry -> off (avoids opentelemetry errors, esp. on Kaggle)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_ENABLED"] = "False"

# gemini-3.5-flash-lite uses fixed sampling -> silence the "temperature ignored" notice
warnings.filterwarnings("ignore", message=".*fixed sampling defaults.*")

# your Gemini key: https://aistudio.google.com/apikey
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

Paste your Gemini key (or load it from Secrets). `gemini-3.5-flash-lite` is the **generation**
model; retrieval uses the local embedding model in the next section.

## 4. Embedding model + load the PDF

In [ ]:
# small, fast, local embedding model (CPU is fine) - used to index the CHILD chunks
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

`all-MiniLM-L6-v2` - tiny (~90 MB), 384-dim, CPU-friendly. It embeds only the **child** chunks.

In [ ]:
# --- small.pdf : VIT FFCS academic regulations write-up ---
# Colab : upload small.pdf via the Files panel            -> "/content/small.pdf"
# Kaggle: Add Input -> your dataset with the file         -> "/kaggle/input/<slug>/small.pdf"
PDF_PATH = "/content/small.pdf"

import os
assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH!r} - upload small.pdf and fix PDF_PATH"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print("loaded", len(documents), "pages")

Set `PDF_PATH` for your environment. `documents` = one `Document` per PDF page.

## 5. Build the ParentDocumentRetriever

* **parent_splitter** -> bigger chunks (900 chars); returned to the LLM as context.
* **child_splitter**  -> small chunks (250 chars); embedded and searched.
* **vectorstore** (Chroma, in-memory) -> holds the child-chunk vectors.
* **docstore** (InMemoryStore) -> holds the parent chunks, keyed by id.

On `add_documents`: each page is split into parents, each parent into children; the
children are embedded into Chroma, each carrying a link back to its parent id.

In [ ]:
# small.pdf is only ~3 pages, so keep the chunks modest
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=0)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)

vectorstore = Chroma(
    collection_name="ffcs_split_parents",
    embedding_function=embedding_function,
)
store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

This cell only **wires the pieces** - nothing is indexed yet:
- `parent_splitter` (900 chars) -> the chunks returned to the LLM as context
- `child_splitter` (250 chars) -> the chunks that get embedded and searched
- `vectorstore` (Chroma) -> holds the child vectors
- `store` (`InMemoryStore`) -> holds the parent chunks, keyed by id
- `ParentDocumentRetriever` -> ties child-id -> parent-id together

In [ ]:
retriever.add_documents(documents)   # split -> embed children -> store parents

parent_ids = list(store.yield_keys())
child_count = len(vectorstore.get()["ids"])
print(f"{len(parent_ids)} parent chunks stored  |  {child_count} child vectors indexed")

`add_documents` runs the full pipeline: each page -> parents -> children; children are embedded
into Chroma (each tagged with its parent id), parents are saved in the docstore.

In [ ]:
# a match on a small child chunk returns the WHOLE parent chunk it belongs to
hits = retriever.invoke("How are students admitted to VIT's B.Tech programmes?")
print(f"{len(hits)} parent chunk(s) returned. First one:\n")
print(hits[0].page_content[:700], "...")

The query matches a small **child** chunk, but `.invoke` returns the **whole parent** that child
belongs to - far more context for the LLM than the raw child would give.

## 6. A RAG chain on top of the retriever

In [ ]:
def format_docs(docs):
    # join the retrieved parent chunks into clean text for the {context} slot
    return "\n\n---\n\n".join(d.page_content for d in docs)

gen_template = (
    "You are a precise assistant answering questions about VIT University's FFCS academic\n"
    "regulations document. Answer ONLY from the context below. If the answer is not in the\n"
    "context, say so.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
)
gen_prompt = ChatPromptTemplate.from_template(gen_template)

# question -> {"context": formatted parent chunks, "question": original question}
setup_and_retrieval = RunnableParallel(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
)

parent_child_rag_chain = setup_and_retrieval | gen_prompt | llm | StrOutputParser()

`format_docs` joins the retrieved parent chunks into one text block. `RunnableParallel` builds
`{context: retriever | format_docs, question: passthrough}`, then `gen_prompt | llm | StrOutputParser`.
The prompt tells the model to answer **only** from the context.

In [ ]:
print(parent_child_rag_chain.invoke("What is FFCS and when did VIT introduce it?"))

## 7. Try several questions about the document

In [ ]:
questions = [
    "What does CAL stand for, and which FFCS regulation version introduced it?",
    "What is VTOP and what is it used for?",
    "How is admission to B.Tech Fashion Technology different from the other B.Tech programmes?",
    "What is V-SIGN and what programmes does it offer?",
    "What lab and studio facilities have been established at V-SIGN?",
]

for q in questions:
    print("=" * 90)
    print("Q:", q)
    print("-" * 90)
    print(parent_child_rag_chain.invoke(q))
    print()

Each answer is grounded in the parent chunks retrieved for that specific question.

## 8. Optional: a tiny interactive chat loop

`input()` works in both Colab and Kaggle notebooks. Type **STOP** to exit.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def ffcs_chat():
    print("Ask me about VIT's FFCS academic regulations. Type STOP to exit.")
    print("-" * 80)
    question = input()
    while question.strip().upper() != "STOP":
        print(parent_child_rag_chain.invoke(question))
        print("\nAnything else?")
        print("-" * 80)
        question = input()

# ffcs_chat()   # <- uncomment to run